In [ ]:
#@title Colab Setup Environment

try:
    import google.colab
    !mkdir -p repository && cd repository && \
     git clone https://github.com/safety-research/circuit-tracer && \
     curl -LsSf https://astral.sh/uv/install.sh | sh && \
     uv pip install -e circuit-tracer/

    import sys
    from huggingface_hub import notebook_login
    sys.path.append('repository/circuit-tracer')
    sys.path.append('repository/circuit-tracer/demos')
    notebook_login(new_session=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
from circuit_tracer import ReplacementModel, attribute
import torch
from tqdm import tqdm

In [ ]:
model_name = 'google/gemma-2-2b'
transcoder_name = "gemma"
model = ReplacementModel.from_pretrained(model_name, transcoder_name, device='cpu', dtype=torch.bfloat16)

In [ ]:
# "What is the capital of the state containing Dallas?\nLet's think step by step.\nThe first step is to find the state containing Dallas.\nThe state containing Dallas is Texas.\nThe second step is to find the capital of Texas.\nThe capital of Texas is"
prompt = "The capital of Utah is Salt"
max_n_logits = 10
desired_logit_prob = 0.95
max_feature_nodes = 8192
batch_size=256
offload='disk' if IN_COLAB else 'cpu'
verbose = True

In [ ]:
graph = attribute(
    prompt=prompt,
    model=model,
    max_n_logits=max_n_logits,
    desired_logit_prob=desired_logit_prob,
    batch_size=batch_size,
    max_feature_nodes=max_feature_nodes,
    offload=offload,
    verbose=verbose
)

# Influence Pruning

In [ ]:
# " Austin"
answer = " Lake"
answer_idx = model.tokenizer(answer).input_ids[-1]

In [ ]:
def metric_fn(logits):
    logits = logits.squeeze()[-1]
    answer_logit = logits[answer_idx]
    mean_top_10 = torch.topk(logits, 10).values.mean()
    return answer_logit - mean_top_10

In [ ]:
with torch.inference_mode():
    full_logits = model(prompt)
    full_metric = metric_fn(full_logits).item()
    empty_logits = model.graph_ablation(
        input=prompt,
        selected_features=[],
        selected_errors=[],
        mean_ablate=False,
        mean_ablation_samples=1000,
        retain_bos_features=True,
        direct_effects=True
    )
    empty_metric = metric_fn(empty_logits).item()
    

In [ ]:
from circuit_tracer.graph import prune_graph

def graph_prunings(
    graph,
    node_thresholds,
    edge_threshold = 1.0
):
    results = []
    
    n_features = len(graph.selected_features)
    n_tokens = graph.n_pos
    n_error_nodes = graph.cfg.n_layers * n_tokens

    for threshold in tqdm(node_thresholds):

        prune_result = prune_graph(graph, node_threshold=threshold, edge_threshold=edge_threshold)
        node_mask = prune_result.node_mask

        feature_mask = node_mask[:n_features]
        selected_feature_indices = torch.where(feature_mask)[0]
        active_feature_indices = graph.selected_features[selected_feature_indices]
        feature_nodes = graph.active_features[active_feature_indices].tolist()
        
        error_mask = node_mask[n_features : n_features + n_error_nodes]
        error_indices = torch.where(error_mask)[0]
        error_nodes = []
        for flat_idx in error_indices:
            layer = flat_idx.item() // n_tokens
            pos = flat_idx.item() % n_tokens
            error_nodes.append((layer, pos))
            
        result_entry = {
            'feature_nodes': feature_nodes,
            'error_nodes': error_nodes,
        }
        results.append(result_entry)

    return results


In [ ]:
import numpy as np

node_thresholds = np.arange(0.05, 1.0, 0.01)
pruned_graphs = graph_prunings(
    graph=graph,
    node_thresholds=node_thresholds,
)

In [ ]:
circuit_metrics = []
for pruned_graph in tqdm(pruned_graphs):
    with torch.inference_mode():
        circuit_logits = model.graph_ablation(
            input=prompt,
            selected_features=pruned_graph['feature_nodes'],
            selected_errors=pruned_graph['error_nodes'],
            mean_ablate=False,
            mean_ablation_samples=1000,
            retain_bos_features=True,
            direct_effects=True
        )
    circuit_metrics.append(metric_fn(circuit_logits).item())


In [ ]:
faithfulness = []
for metric in circuit_metrics:
    faithfulness.append((metric - empty_metric) / (full_metric - empty_metric))

In [ ]:
faithfulness

In [ ]:
graph_size = []
for pruned_graph in pruned_graphs:
    graph_size.append(len(pruned_graph['feature_nodes']) + len(pruned_graph['error_nodes']))

In [ ]:
graph_size